# Resistance mutations in aquatic plastisphere microbiomes

All script files are located in `sbatch_files/`. 

Steps:
1. Download the fastq data (specified in *download.txt*) using `download.sh` 
2. Run Pipeline 1, see TODO_URL_TO_IT for more details. TODO: Ok to link to it or jsut describe it?
3. Download the card database using `download_card.sh`
4. Build the database for MuMaMe using `mumame_build.sh`
5. Run MuMaMe using `split_mumame.sh`, it will be grouped according to *beginning_filenames.txt*, in order to speed up the operation.
6. The files *\*.table.txt* and *\*.per-mutation.txt* will be used below.

# Data analysis

In [36]:
# Load Libraries
library("tidyverse")

# List all relevant files
all_files_list <- list.files(path = "Mumame_results", pattern = "*.per-mutation.txt", full.names = TRUE) 
# See also pattern = "*.table.txt"
# per-mutation splits the Accession into one line per mutation, and uses the same hits for all. Easier with splitting below but result feels it could be wrong?

# Apply read_tsv to all elements of all_files
data_list <- all_files_list %>%
  map(~ read_tsv(., show_col_types = FALSE))

combined_data <- reduce(data_list, function(x, y) {
  full_join(x, y, by = "Accession")
})

#combined_data <- combined_data %>% 
#    replace(is.na(.), 0)

#combined_data

# Load in the metadata tables
snps <- read_tsv("card-data/snps.txt", show_col_types = FALSE)
aro_index <- read_tsv("card-data/aro_index.tsv", show_col_types = FALSE)
aro_categories <- read_tsv("card-data/aro_categories.tsv", show_col_types = FALSE)
aro_categories_index <- read_tsv("card-data/aro_categories_index.tsv", show_col_types = FALSE)

In [37]:
# Show one row which contain data for both samples
# For per-mutation.txt
combined_data[which(combined_data["Accession"] == "A67T:Staphylococcus aureus fusA with mutation conferring resistance to fusidic acid"), c(1:5, 39:63, 1100:1103)]
combined_data[which(combined_data["Accession"] == "V90A:Staphylococcus aureus fusA with mutation conferring resistance to fusidic acid"), c(1:5, 39:63, 1100:1103)]
# Below for table.txt
#combined_data[which(combined_data["Accession"] == "A67T+V90A:Staphylococcus aureus fusA with mutation conferring resistance to fusidic acid"), c(1:5, 39:63, 1100:1103)]

# Print the metadata tables
#print("snps")
#snps
#print("aro_index")
#aro_index
#print("aro_categories")
#aro_categories
#print("aro_categories_index")
#aro_categories_index


Accession,DRR528401_1_val-Mut,DRR528401_2_val-Mut,DRR528402_1_val-Mut,DRR528402_2_val-Mut,ERR10466833_2_val-Mut,ERR10466834_1_val-Mut,ERR10466834_2_val-Mut,ERR10466835_1_val-Mut,ERR10466835_2_val-Mut,⋯,ERR10466836_1_val-WT,ERR10466836_2_val-WT,ERR10466837_1_val-WT,ERR10466837_2_val-WT,ERR10466838_1_val-WT,ERR10466838_2_val-WT,SRR29811116_1_val-WT,SRR29811116_2_val-WT,SRR29811117_1_val-WT,SRR29811117_2_val-WT
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A67T:Staphylococcus aureus fusA with mutation conferring resistance to fusidic acid,0,0,0,1,0,0,3,0,1,⋯,1,0,0,0,0,0,0,0,0,0


Accession,DRR528401_1_val-Mut,DRR528401_2_val-Mut,DRR528402_1_val-Mut,DRR528402_2_val-Mut,ERR10466833_2_val-Mut,ERR10466834_1_val-Mut,ERR10466834_2_val-Mut,ERR10466835_1_val-Mut,ERR10466835_2_val-Mut,⋯,ERR10466836_1_val-WT,ERR10466836_2_val-WT,ERR10466837_1_val-WT,ERR10466837_2_val-WT,ERR10466838_1_val-WT,ERR10466838_2_val-WT,SRR29811116_1_val-WT,SRR29811116_2_val-WT,SRR29811117_1_val-WT,SRR29811117_2_val-WT
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
V90A:Staphylococcus aureus fusA with mutation conferring resistance to fusidic acid,0,0,0,1,0,0,0,0,1,⋯,1,0,0,0,0,0,0,0,0,0


In [38]:
# 1. Mumame -> shortname
# 2. shortname -> snps.txt -> aro number
# 3. aro number -> aro_index.tsv -> categories

# Extract mutation names (Note "Wildtype" in some)
combined_data$Mutations <- combined_data$Accession %>%
    str_split_i(":", i = 1)
# Split further? Not needed if use per-mutation
#combined_data$mutations %>%
#    str_split("\\+")

# Rename Accession in combined data to avoid confusion with ARO
combined_data <- combined_data %>%
    rename(Description = Accession)

# Add the metadata to the dataframe
combined_data <- left_join(combined_data, snps, by = "Mutations")
combined_data[which(combined_data$Mutations == "G124S"), ]

snps[29, ]
combined_data[8, ]

Warning message in left_join(combined_data, snps, by = "Mutations"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 8 of `x` matches multiple rows in `y`.
ℹ Row 29 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


Description,DRR528401_1_val-Mut,DRR528401_2_val-Mut,DRR528402_1_val-Mut,DRR528402_2_val-Mut,DRR528403_1_val-Mut,DRR528403_2_val-Mut,DRR528404_1_val-Mut,DRR528404_2_val-Mut,DRR528405_1_val-Mut,⋯,SRR8369863_1_val-WT,SRR8369863_2_val-WT,Mutations,Accession,Name,Model Type,Parameter Type,CARD Short Name,source,citation
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
G124S:Mycobacterium tuberculosis katG mutations conferring resistance to isoniazid,2,0,2,1,1,0,1,0,0,⋯,7,1,G124S,3003302,Bartonella bacilliformis gyrB conferring resistance to aminocoumarin,protein variant model,single resistance variant,Bbac_gyrA_AMU,Curated-R,9797224
G124S:Mycobacterium tuberculosis katG mutations conferring resistance to isoniazid,2,0,2,1,1,0,1,0,0,⋯,7,1,G124S,3003392,Mycobacterium tuberculosis katG mutations conferring resistance to isoniazid,protein variant model,single resistance variant,Mtub_katG_INH,WHO-R,36635309
G124S:Mycobacterium tuberculosis katG mutations conferring resistance to isoniazid,2,0,2,1,1,0,1,0,0,⋯,7,1,G124S,3003394,Mycobacterium tuberculosis pncA mutations conferring resistance to pyrazinamide,protein variant model,single resistance variant,Mtub_pncA_PZA,Curated-R,35944069
G124S:Bartonella bacilliformis gyrB conferring resistance to aminocoumarin,2,7,7,9,0,0,8,23,14,⋯,0,0,G124S,3003302,Bartonella bacilliformis gyrB conferring resistance to aminocoumarin,protein variant model,single resistance variant,Bbac_gyrA_AMU,Curated-R,9797224
G124S:Bartonella bacilliformis gyrB conferring resistance to aminocoumarin,2,7,7,9,0,0,8,23,14,⋯,0,0,G124S,3003392,Mycobacterium tuberculosis katG mutations conferring resistance to isoniazid,protein variant model,single resistance variant,Mtub_katG_INH,WHO-R,36635309
G124S:Bartonella bacilliformis gyrB conferring resistance to aminocoumarin,2,7,7,9,0,0,8,23,14,⋯,0,0,G124S,3003394,Mycobacterium tuberculosis pncA mutations conferring resistance to pyrazinamide,protein variant model,single resistance variant,Mtub_pncA_PZA,Curated-R,35944069


Accession,Name,Model Type,Parameter Type,Mutations,CARD Short Name,source,citation
<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
3003302,Bartonella bacilliformis gyrB conferring resistance to aminocoumarin,protein variant model,single resistance variant,G124S,Bbac_gyrA_AMU,Curated-R,9797224


Description,DRR528401_1_val-Mut,DRR528401_2_val-Mut,DRR528402_1_val-Mut,DRR528402_2_val-Mut,DRR528403_1_val-Mut,DRR528403_2_val-Mut,DRR528404_1_val-Mut,DRR528404_2_val-Mut,DRR528405_1_val-Mut,⋯,SRR8369863_1_val-WT,SRR8369863_2_val-WT,Mutations,Accession,Name,Model Type,Parameter Type,CARD Short Name,source,citation
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
A381V:Mycobacterium tuberculosis rpoB with mutations conferring resistance to rifampicin,8,1,1,0,0,6,0,0,0,⋯,7,9,A381V,3003283,Mycobacterium tuberculosis rpoB with mutations conferring resistance to rifampicin,protein variant model,single resistance variant,Mtub_rpoB_RIF,WHO-R,8870258
